# Chunkwise CSE · Docling extraction on GPU

Run from VS Code with the Google Colab extension. Full steps are in the README.

1. Keep the annual report PDFs in `Annual Reports`, in their company sub-folders. Drive for Desktop syncs that folder to `Othercomputers/My laptop/Lectures/Year 2/DSGP/Annual Reports`
2. **Select Kernel → Colab → New Colab Server → GPU**
3. **Run All**, and approve the Google Drive prompt when it appears
4. Back in this project, import the results with `docling-pipeline run --config configs/extraction.yaml`

Folders named in `EXCLUDE_DIRS` are skipped, which keeps interim reports out, as `parse.exclude_dirs` does in the project config.

For each report, `MyDrive/chunkwise-cse/extracted_<layout model>` gets:

| File | What it holds |
|---|---|
| `<report>.json` | Docling JSON: text, headings, tables, pictures, page numbers (input for the chunker) |
| `<report>.md` | Markdown version for reading |
| `<report>_pictures.json` | Every picture: page, type (bar chart, logo, photo…), title, image file, and for charts the values read from the labels printed on the chart |
| `<report>_pictures/` | A PNG of every picture |

Reports that are already done are skipped, so after a disconnect just run all again. A report that has a JSON but no `_pictures.json` only gets its pictures added.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U docling

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU attached: connect a Colab server with a GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

LAYOUT_MODEL = "layout_egret_large"
TABLEFORMER_V2 = False
TABLE_CELL_MATCHING = True
SAVE_PICTURE_IMAGES = True
PICTURE_IMAGE_SCALE = 2.0
LAYOUT_BATCH_SIZE = 32
QUEUE_MAX_SIZE = 32
EXCLUDE_DIRS = {"interim"}

INPUT_DIR = Path("/content/drive/Othercomputers/My laptop/Lectures/Year 2/DSGP/Annual Reports")
OUTPUT_DIR = Path(f"/content/drive/MyDrive/chunkwise-cse/extracted_{LAYOUT_MODEL.removeprefix('layout_')}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(
    path for path in INPUT_DIR.rglob("*.pdf")
    if not EXCLUDE_DIRS & {part.lower() for part in path.relative_to(INPUT_DIR).parts[:-1]}
)
print(f"{len(pdfs)} PDFs found, writing to {OUTPUT_DIR}")

In [ ]:
import logging
import time

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    HeadingHierarchyOptions,
    LayoutObjectDetectionOptions,
    TableFormerMode,
    TableStructureOptions,
    TableStructureV2Options,
    ThreadedPdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.threaded_standard_pdf_pipeline import ThreadedStandardPdfPipeline

logging.getLogger("docling").setLevel(logging.WARNING)

table_options = (
    TableStructureV2Options(do_cell_matching=TABLE_CELL_MATCHING)
    if TABLEFORMER_V2
    else TableStructureOptions(mode=TableFormerMode.ACCURATE, do_cell_matching=TABLE_CELL_MATCHING)
)

pipeline_options = ThreadedPdfPipelineOptions(
    accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA),
    do_ocr=False,
    do_table_structure=True,
    layout_options=LayoutObjectDetectionOptions.from_preset(LAYOUT_MODEL),
    table_structure_options=table_options,
    do_picture_classification=True,
    heading_hierarchy_options=HeadingHierarchyOptions(enabled=True),
    layout_batch_size=LAYOUT_BATCH_SIZE,
    table_batch_size=4,
    queue_max_size=QUEUE_MAX_SIZE,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=ThreadedStandardPdfPipeline,
            pipeline_options=pipeline_options,
        )
    }
)

start = time.time()
converter.initialize_pipeline(InputFormat.PDF)
print(f"Pipeline ready in {time.time() - start:.1f}s")

In [ ]:
import json
import re
from statistics import mean, median

import pypdfium2 as pdfium
from docling_core.types.doc import DoclingDocument, PictureItem, TextItem

PERIOD = re.compile(
    r"^(FY\s?\d{2,4}|(19|20)\d{2}(/\d{2,4})?|[QH][1-4]\s?(FY\s?)?\d{0,4}|"
    r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s?\d{0,4})$",
    re.IGNORECASE,
)
VALUE = re.compile(r"^\(?[-–]?\d[\d,]*(\.\d+)?\)?\s?%?\**$")
UNIT_LABEL = re.compile(r"^\(.+\)$")
CROP_PAD_PT = 4


def parse_value(text):
    text = text.strip()
    if not VALUE.match(text) or PERIOD.match(text):
        return None
    value = float(re.sub(r"[^\d.]", "", text))
    return -value if text[0] in "(-–" else value


def group(items, key, tol):
    groups = []
    for item in sorted(items, key=lambda i: i[key]):
        if groups and item[key] - groups[-1][-1][key] <= tol:
            groups[-1].append(item)
        else:
            groups.append([item])
    return groups


def series_unit(printed, values, axes):
    if any(p and "%" in p for p in printed):
        return "%"
    nums = [v for v in values if v is not None]
    fits = [
        ax for ax in axes
        if ax["low"] - 0.05 * (ax["high"] - ax["low"]) <= min(nums)
        and max(nums) <= ax["high"] + 0.05 * (ax["high"] - ax["low"])
    ]
    return min(fits, key=lambda ax: ax["high"] - ax["low"])["unit"] if fits else None


def read_axis(items, a, b):
    periods = [i for i in items if PERIOD.match(i["text"])]
    rows = [r for r in group(periods, b, 4) if len(r) >= 2]
    if not rows:
        return None
    cats = sorted(max(rows, key=len), key=lambda i: i[a])
    pos = [c[a] for c in cats]
    spacing = median(q - p for p, q in zip(pos, pos[1:]))
    if spacing <= 0:
        return None
    names = [c["text"] for c in cats]
    cat_b = median(c[b] for c in cats)
    used = {id(c) for c in cats}

    for i in items:
        if id(i) in used or parse_value(i["text"]) is not None or PERIOD.match(i["text"]):
            continue
        for k, c in enumerate(cats):
            if abs(i[a] - c[a]) <= 0.3 * spacing and 0 < i[b] - c[b] <= 9:
                names[k] += " " + i["text"]
                used.add(id(i))

    numbers = [
        dict(i, value=parse_value(i["text"]))
        for i in items
        if id(i) not in used and parse_value(i["text"]) is not None and abs(i[b] - cat_b) > 4
    ]

    axes, values = [], []
    for col in group(numbers, a, 3):
        col_a = mean(n[a] for n in col)
        if len(col) >= 3 and min(abs(col_a - p) for p in pos) > 0.3 * spacing:
            top = min(n[b] for n in col)
            unit = next(
                (
                    i["text"] for i in items
                    if id(i) not in used
                    and parse_value(i["text"]) is None
                    and UNIT_LABEL.match(i["text"])
                    and abs(i[a] - col_a) <= 15
                    and 0 < top - i[b] <= 30
                ),
                None,
            )
            ticks = [n["value"] for n in col]
            axes.append({"unit": unit, "low": min(ticks), "high": max(ticks)})
        else:
            values.extend(col)

    placed = []
    for v in values:
        k = min(range(len(pos)), key=lambda k: abs(v[a] - pos[k]))
        if abs(v[a] - pos[k]) <= 0.45 * spacing:
            placed.append(dict(v, cat=k))

    if not placed:
        return {"status": "values_not_printed", "categories": names, "axes": axes, "series": [], "unplaced": []}

    need = max(2, -(-len(cats) * 3 // 5))
    series, rest = [], []
    for row in group(placed, b, 3):
        ks = [v["cat"] for v in row]
        if len(row) >= need and len(set(ks)) == len(ks):
            series.append(row)
        else:
            rest.extend(row)

    per_cat = {}
    for v in sorted(rest, key=lambda v: v[b]):
        per_cat.setdefault(v["cat"], []).append(v)
    counts = {len(vs) for vs in per_cat.values()}
    unplaced = []
    if len(counts) == 1:
        for r in range(counts.pop()):
            series.append([vs[r] for vs in per_cat.values()])
    else:
        unplaced = [
            {"category": names[k], "printed": [v["text"] for v in vs]}
            for k, vs in sorted(per_cat.items())
        ]

    out = []
    for s in sorted(series, key=lambda s: mean(v[b] for v in s)):
        values_row = [None] * len(cats)
        printed_row = [None] * len(cats)
        for v in s:
            values_row[v["cat"]] = v["value"]
            printed_row[v["cat"]] = v["text"]
        out.append({
            "unit": series_unit(printed_row, values_row, axes),
            "values": values_row,
            "printed": printed_row,
        })
    return {"status": "values_printed", "categories": names, "axes": axes, "series": out, "unplaced": unplaced}


def read_chart(items):
    return read_axis(items, "x", "y") or read_axis(items, "y", "x")


def chart_markdown(chart):
    if not chart or chart["status"] != "values_printed":
        return None
    lines = [
        "| | " + " | ".join(chart["categories"]) + " |",
        "|---" * (len(chart["categories"]) + 1) + "|",
    ]
    for n, s in enumerate(chart["series"], 1):
        cells = [p if p is not None else "" for p in s["printed"]]
        lines.append(f"| {s['unit'] or f'series {n}'} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def picture_texts(doc, pic):
    seen, items = set(), []
    for item, _ in doc.iterate_items(root=pic, traverse_pictures=True):
        if not isinstance(item, TextItem) or not item.prov or not item.text.strip():
            continue
        prov = item.prov[0]
        box = prov.bbox.to_top_left_origin(doc.pages[prov.page_no].size.height)
        x, y = (box.l + box.r) / 2, (box.t + box.b) / 2
        key = (item.text.strip(), round(x), round(y))
        if key not in seen:
            seen.add(key)
            items.append({"text": key[0], "x": x, "y": y})
    return items


def picture_class(pic):
    meta = getattr(pic, "meta", None)
    if meta is None or meta.classification is None:
        return None, None
    best = meta.classification.get_main_prediction()
    return best.class_name, best.confidence


def text_before(order, index, page_no):
    if index == 0:
        return None
    item = order[index - 1]
    if (
        isinstance(item, TextItem)
        and item.prov
        and item.prov[0].page_no == page_no
        and len(item.text) <= 120
        and parse_value(item.text) is None
    ):
        return item.text
    return None


def crop_pictures(pdf_path, doc, pictures, image_dir, scale):
    image_dir.mkdir(parents=True, exist_ok=True)
    by_page = {}
    for n, pic in enumerate(pictures, 1):
        by_page.setdefault(pic.prov[0].page_no, []).append((n, pic))
    paths = {}
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for page_no, page_pictures in by_page.items():
            image = pdf[page_no - 1].render(scale=scale).to_pil()
            height = doc.pages[page_no].size.height
            for n, pic in page_pictures:
                box = pic.prov[0].bbox.to_top_left_origin(height)
                left = max(0, round((box.l - CROP_PAD_PT) * scale))
                top = max(0, round((box.t - CROP_PAD_PT) * scale))
                right = min(image.width, round((box.r + CROP_PAD_PT) * scale))
                bottom = min(image.height, round((box.b + CROP_PAD_PT) * scale))
                if right <= left or bottom <= top:
                    continue
                path = image_dir / f"p{page_no:03d}_{n:03d}.png"
                image.crop((left, top, right, bottom)).save(path)
                paths[id(pic)] = path
    finally:
        pdf.close()
    return paths


def save_pictures(doc, pdf_path, pictures_out, save_images, scale):
    order = [item for item, _ in doc.iterate_items()]
    indexed = [(n, item) for n, item in enumerate(order) if isinstance(item, PictureItem) and item.prov]
    pictures = [pic for _, pic in indexed]
    image_dir = pictures_out.with_suffix("")
    images = crop_pictures(pdf_path, doc, pictures, image_dir, scale) if save_images else {}

    records = []
    for n, pic in indexed:
        prov = pic.prov[0]
        box = prov.bbox.to_top_left_origin(doc.pages[prov.page_no].size.height)
        items = picture_texts(doc, pic)
        chart = read_chart(items)
        kind, confidence = picture_class(pic)
        image = images.get(id(pic))
        records.append({
            "id": pic.self_ref,
            "page": prov.page_no,
            "bbox": [round(box.l, 1), round(box.t, 1), round(box.r, 1), round(box.b, 1)],
            "class": kind,
            "confidence": round(confidence, 3) if confidence is not None else None,
            "caption": pic.caption_text(doc) or None,
            "title_guess": text_before(order, n, prov.page_no),
            "image": str(image.relative_to(pictures_out.parent)) if image else None,
            "labels": [
                i["text"] for i in sorted(items, key=lambda i: (round(i["y"]), i["x"]))
                if parse_value(i["text"]) is None
            ],
            "chart": chart,
            "markdown": chart_markdown(chart),
        })

    pictures_out.write_text(json.dumps(records, indent=2, ensure_ascii=False))
    printed = sum(1 for r in records if r["chart"] and r["chart"]["status"] == "values_printed")
    unprinted = sum(1 for r in records if r["chart"] and r["chart"]["status"] == "values_not_printed")
    return len(records), printed, unprinted

In [ ]:
import csv
import gc

import psutil

log_path = OUTPUT_DIR / "extraction_log.csv"
write_header = not log_path.exists()

with open(log_path, "a", newline="") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["file", "status", "pages", "seconds", "pages_per_second"])
    for i, pdf in enumerate(pdfs, 1):
        rel = pdf.relative_to(INPUT_DIR)
        json_out = (OUTPUT_DIR / rel).with_suffix(".json")
        pictures_out = json_out.with_name(f"{json_out.stem}_pictures.json")
        if json_out.exists() and pictures_out.exists():
            continue
        json_out.parent.mkdir(parents=True, exist_ok=True)

        if json_out.exists():
            doc = DoclingDocument.load_from_json(json_out)
            print(f"[{i}/{len(pdfs)}] {rel}: already converted, adding pictures")
        else:
            start = time.time()
            result = converter.convert(pdf, raises_on_error=False)
            seconds = time.time() - start
            pages = len(result.pages)
            rate = pages / seconds if seconds else 0.0
            status = result.status
            doc = None
            if status in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
                doc = result.document
                doc.save_as_json(json_out)
                doc.save_as_markdown(json_out.with_suffix(".md"))
            del result
            writer.writerow([str(rel), status.value, pages, round(seconds, 1), round(rate, 2)])
            f.flush()
            print(f"[{i}/{len(pdfs)}] {rel}: {status.value}, {pages} pages, {rate:.2f} pages/s")

        if doc is not None:
            total, printed, unprinted = save_pictures(
                doc, pdf, pictures_out, SAVE_PICTURE_IMAGES, PICTURE_IMAGE_SCALE
            )
            print(f"    {total} pictures: {printed} charts with printed values, {unprinted} charts without")
        del doc
        gc.collect()
        ram = psutil.virtual_memory()
        print(f"    RAM {ram.used / 1e9:.1f}/{ram.total / 1e9:.1f} GB")

## Check one report's tables

In [ ]:
sample = next(p for p in OUTPUT_DIR.rglob("*.json") if not p.stem.endswith("_pictures"))
doc = DoclingDocument.load_from_json(sample)
print(f"{sample.name}: {len(doc.texts)} text items, {len(doc.tables)} tables, {len(doc.pictures)} pictures")
if doc.tables:
    display(doc.tables[0].export_to_dataframe(doc=doc).head(10))

## Check one report's pictures and charts

In [ ]:
import pandas as pd
from IPython.display import Image

pictures_file = next(OUTPUT_DIR.rglob("*_pictures.json"))
records = json.loads(pictures_file.read_text())
overview = pd.DataFrame([
    {
        "class": r["class"] or "unclassified",
        "chart": r["chart"]["status"] if r["chart"] else "no chart axis",
    }
    for r in records
])
print(f"{pictures_file.name}: {len(records)} pictures")
display(overview.value_counts().rename("count").reset_index())

charts = [r for r in records if r["markdown"]]
print(f"\n{len(charts)} charts with printed values, first 5:")
for r in charts[:5]:
    print(f"\npage {r['page']} · {r['caption'] or r['title_guess'] or 'untitled'}")
    if r["image"]:
        display(Image(filename=str(pictures_file.parent / r["image"]), width=500))
    print(r["markdown"])